In [ ]:
# creating spark session
exec(open('/home/jovyan/.ipython/profile_default/startup/00-spark-session.py').read())

In [ ]:
# Adding Display functionality of Databricks 
exec(open('/home/jovyan/.ipython/profile_default/startup/01-databricks-utils.py').read())

In [16]:
from pyspark.sql.functions import *

# Average Experience Per Project

## Problem Description

You are given two DataFrames:

- **`project`**:
  - `project_id` (int): Project identifier
  - `employee_id` (int): Employee assigned to the project

- **`employee`**:
  - `employee_id` (int): Unique employee identifier
  - `name` (str): Employee name
  - `experience_years` (int): Years of experience

**Task:**  
Report the **average experience years** of all employees for each project, rounded to **2 decimal places**.

**Output:**  
Return a DataFrame with columns:
- `project_id`
- `average_years` – average experience rounded to 2 decimals

Sorted by `project_id` in ascending order.

---

## Example

**Input Data**

`project`:

| project_id | employee_id |
|------------|-------------|
| 1          | 1           |
| 1          | 2           |
| 1          | 3           |
| 2          | 1           |
| 2          | 4           |

`employee`:

| employee_id | name    | experience_years |
|-------------|---------|------------------|
| 1           | Alice   | 3                |
| 2           | Bob     | 5                |
| 3           | Charlie | 7                |
| 4           | Diana   | 2                |

**Expected Output:**

| project_id | average_years |
|------------|---------------|
| 1          | 5.00          |
| 2          | 2.50          |

- Project 1: (3 + 5 + 7) / 3 = 5.00
- Project 2: (3 + 2) / 2 = 2.50

---



In [1]:
## Dataset Generation

from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Create a Spark session
spark = SparkSession.builder.appName("AvgExperiencePerProject").getOrCreate()

# Define schema for project
project_schema = StructType([
    StructField("project_id", IntegerType(), True),
    StructField("employee_id", IntegerType(), True)
])

project_data = [
    (1, 1),
    (1, 2),
    (1, 3),
    (2, 1),
    (2, 4)
]

project_df = spark.createDataFrame(project_data, schema=project_schema)

# Define schema for employee
employee_schema = StructType([
    StructField("employee_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("experience_years", IntegerType(), True)
])

employee_data = [
    (1, "Alice", 3),
    (2, "Bob", 5),
    (3, "Charlie", 7),
    (4, "Diana", 2)
]

employee_df = spark.createDataFrame(employee_data, schema=employee_schema)

project_df.show()
employee_df.show()

+----------+-----------+
|project_id|employee_id|
+----------+-----------+
|         1|          1|
|         1|          2|
|         1|          3|
|         2|          1|
|         2|          4|
+----------+-----------+

+-----------+-------+----------------+
|employee_id|   name|experience_years|
+-----------+-------+----------------+
|          1|  Alice|               3|
|          2|    Bob|               5|
|          3|Charlie|               7|
|          4|  Diana|               2|
+-----------+-------+----------------+



# Using Spark SQL

In [3]:
project_df.createOrReplaceTempView("project")
employee_df.createOrReplaceTempView("emp")

In [10]:
spark.sql(
    """
    SELECT p.project_id,ROUND(AVG(e.experience_years),2) as avg_years 
    from emp e 
    join project p on p.employee_id = e.employee_id 
    group by p.project_id
    """
).show()

+----------+---------+
|project_id|avg_years|
+----------+---------+
|         1|      5.0|
|         2|      2.5|
+----------+---------+



# Using PySpark

In [20]:
from pyspark.sql.functions import avg, round

result = (
    project_df.join(employee_df, on="employee_id")
    .groupBy("project_id")
    .agg(round(avg("experience_years"), 2).alias("average_years"))
    .orderBy("project_id")
)
result.show()

+----------+-------------+
|project_id|average_years|
+----------+-------------+
|         1|          5.0|
|         2|          2.5|
+----------+-------------+

